In [2]:
# ============================================================================
# Notebook: 02_recode.ipynb
#
# Verification Channels and Remote Work — recode raw -> analytic samples
#
# Reads data/raw/survey_{2024,2025}.csv, applies the common analytic schema
# (via src/recode.py), and writes three analytic samples to data/processed/:
#   analysis_2024.csv         (main sample)
#   analysis_2025.csv         (main sample)
#   analysis_2025_robust.csv  (robustness: "Your choice" -> Remote)
#
# Runs from the notebooks/ folder; project root is one level up.
# ============================================================================


# %%
# ---------------------------------------------------------------------------
# Cell 1 | Paths, imports, and make src/ importable
# ---------------------------------------------------------------------------
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Allow "import recode" from src/
sys.path.insert(0, str(ROOT / "src"))
from recode import recode, build_sample, build_robust_2025, CORE, MIN_COUNTRY_N

print("Setup complete. Min country N:", MIN_COUNTRY_N)


# %%
# ---------------------------------------------------------------------------
# Cell 2 | Load raw and apply the analytic schema
# ---------------------------------------------------------------------------
df24 = pd.read_csv(RAW_DIR / "survey_2024.csv", low_memory=False)
df25 = pd.read_csv(RAW_DIR / "survey_2025.csv", low_memory=False)

r24 = recode(df24, 2024)
r25 = recode(df25, 2025)
print("Recoded raw shapes:", r24.shape, r25.shape)


# %%
# ---------------------------------------------------------------------------
# Cell 3 | Build analytic samples
# ---------------------------------------------------------------------------
s24 = build_sample(r24)
s25 = build_sample(r25)
r25_rob = build_robust_2025(r25)

for label, s in [("2024", s24), ("2025", s25), ("2025 robust", r25_rob)]:
    print(f"{label:12s} N={len(s):>6}  countries={s['country'].nunique():>3}  "
          f"jobsat_mean={s['jobsat'].mean():.2f}")


# %%
# ---------------------------------------------------------------------------
# Cell 4 | Quick recode diagnostics (verify categories came through cleanly)
# ---------------------------------------------------------------------------
for label, s in [("2024", s24), ("2025", s25)]:
    print(f"\n=== {label} ===")
    print("remote3:", s["remote3"].value_counts().to_dict())
    print("ai_use :", s["ai_use"].value_counts().to_dict())
    print("devgroup:", s["devgroup"].value_counts().to_dict())

# 2025 frequency (kept on the recoded frame, before non-user fill)
print("\n2025 ai_freq (users only):",
      r25["ai_freq"].value_counts(dropna=False).to_dict())


# %%
# ---------------------------------------------------------------------------
# Cell 5 | Save analytic samples to data/processed/
#
# We keep the columns the analysis needs. For 2025 we also carry ai_freq so the
# exploratory intensity analysis can run without touching raw again.
# ---------------------------------------------------------------------------
cols_main = ["jobsat", "ai_use", "remote3", "country", "workexp", "devgroup", "yourchoice", "year"]

s24[cols_main].to_csv(PROC_DIR / "analysis_2024.csv", index=False)

cols_2025 = cols_main + (["ai_freq"] if "ai_freq" in s25.columns else [])
s25[cols_2025].to_csv(PROC_DIR / "analysis_2025.csv", index=False)

# Robustness sample: attach ai_freq from the recoded 2025 frame by index
r25_rob_out = r25_rob.copy()
if "ai_freq" in r25.columns:
    r25_rob_out["ai_freq"] = r25.loc[r25_rob_out.index, "ai_freq"]
cols_rob = cols_main + (["ai_freq"] if "ai_freq" in r25_rob_out.columns else [])
r25_rob_out[cols_rob].to_csv(PROC_DIR / "analysis_2025_robust.csv", index=False)

print("Saved analytic samples:")
for f in ["analysis_2024.csv", "analysis_2025.csv", "analysis_2025_robust.csv"]:
    p = PROC_DIR / f
    print(f"  {f:28s} {p.stat().st_size/1e6:.2f} MB  ({sum(1 for _ in open(p))-1} rows)")


# %%
# ---------------------------------------------------------------------------
# Cell 6 | Done
# ---------------------------------------------------------------------------
print("Recoding complete.")
print("Next: 03_analysis.ipynb  (multilevel models H1-H5 + replication)")

Setup complete. Min country N: 100
Recoded raw shapes: (65437, 122) (49191, 181)
2024         N= 26576  countries= 48  jobsat_mean=6.95
2025         N= 16081  countries= 36  jobsat_mean=7.21
2025 robust  N= 18876  countries= 40  jobsat_mean=7.23

=== 2024 ===
remote3: {'Hybrid': 11655, 'Remote': 10294, 'In-person': 4627}
ai_use : {1.0: 16426, 0.0: 10150}
devgroup: {'Developer': 20110, 'Other': 2318, 'Data/ML': 1334, 'Manager': 1059, 'Research': 885, 'DevOps/Infra': 731, 'Student': 139}

=== 2025 ===
remote3: {'Hybrid': 7095, 'Remote': 6520, 'In-person': 2466}
ai_use : {1.0: 12896, 0.0: 3185}
devgroup: {'Developer': 12082, 'Other': 1942, 'Data/ML': 905, 'Manager': 472, 'DevOps/Infra': 449, 'Research': 164, 'Student': 67}

2025 ai_freq (users only): {nan: 22722, 3.0: 15883, 2.0: 5958, 1.0: 4628}
Saved analytic samples:
  analysis_2024.csv            1.39 MB  (26576 rows)
  analysis_2025.csv            0.91 MB  (16081 rows)
  analysis_2025_robust.csv     1.06 MB  (18876 rows)
Recoding com